# LABORATORIO 1: Diagnóstico experimental de paralelismo.
INTEGRANTES: Benjamín Aballay / Felipe Martínez \
Asignatura: Computación paralela y distribuida. \
Docente: Michael Miranda \
Sección: 412 \
Institución: Universidad Tecnológica Metropolitana

## INTRODUCCIÓN DEL LABORATORIO

El presente laboratorio tiene como proposito desarrollar el pensamiento crítico y riguroso respecto al rendimiento computacional. A través de la experimentación en Python, se busca identificar la estructura de diferentes problemas para determinar si poseen una naturaleza secuencial o si presentan oportunidades reales de paralelización.

Para este laboratorio, es necesario lograr los siguientes puntos:
- Construir métricas base de tiempo para tareas de distinta complejidad.
- Comparar variantes de procesamiento para visualizar la estructura del problema.
- Evaluar casos de uso aplicados para decidir entre estrategias de computación paralela o distribuida.


## EJERCICIO 1: Medición base de una tarea secuencial simple.

In [ ]:
# Importación de librerias.
import pandas as pd
import time
import math
import sympy as sp
from IPython.display import display, Math
import random
import statistics

In [ ]:
 # 1. Cargar el nuevo set de datos más grande
df = pd.read_csv('numeros_grandes.csv')
datos_grandes = df['Numeros'].tolist()

def suma_cuadrados_desde_lista(lista):
    acumulado = 0
    for numero in lista:
        acumulado += numero * numero
    return acumulado

# 2. Definir tamaños de entrada significativos dentro de los 1,000 datos
tamanos = [100, 500, 1000]
resultados = []

print(f"{'n':>12} | {'Resultado':>25} | {'Tiempo (s)':>12}")
print("-" * 60)

# 3. Medir tiempos de ejecución
for n in tamanos:
    subconjunto = datos_grandes[:n]

    inicio = time.perf_counter()
    resultado = suma_cuadrados_desde_lista(subconjunto)
    fin = time.perf_counter()

    duracion = fin - inicio
    resultados.append((n, resultado, duracion))

    # 4. Registro de resultados
    print(f"{n:12,d} | {resultado:25,d} | {duracion:12.8f}")

           n |                 Resultado |   Tiempo (s)
------------------------------------------------------------
         100 |           341,628,048,552 |   0.00001234
         500 |         1,666,745,433,273 |   0.00003527
       1,000 |         3,455,366,621,986 |   0.00007118


Se observa que el tiempo de ejecución aumenta de manera proporcional al tamaño de la entrada $n$. Por ejemplo, al pasar de $n=500$ a $n=1000$ elementos, el tiempo de procesamiento aproximadamente se duplica, lo que indica una complejidad algorítmica de orden lineal, denotada como $O(n)$.

Los tiempos registrados, aunque extremadamente bajos debido al tamaño reducido de la muestra actual, permiten establecer una línea base de rendimiento para comparar futuras optimizaciones.

Esta implementación constituye una referencia secuencial base por las siguientes razones :
- El programa utiliza un solo flujo de control (un solo núcleo de la CPU), procesando un dato a la vez en el orden estricto en que aparecen en la lista.
- La variable "acumulado" se actualiza de forma iterativa dentro de un ciclo "for". Cada suma depende del valor calculado en el paso inmediatamente anterior, lo que refuerza la estructura serial del algoritmo en esta versión.
- No existe ninguna división de la carga de trabajo ni uso de bibliotecas de paralelismo; el procesador espera a completar la instrucción actual antes de iniciar la siguiente.

## EJERCICIO 2: Comparación de variantes de procesamiento y discusión sobre paralelización.

In [ ]:
# 1. DEFINICIÓN DE LA FUNCIÓN DE ALTA INTENSIDAD
def carga_pesada(x):
    """
    Realiza una transformación costosa sobre cada elemento.
    Simula una tarea de alta intensidad para observar diferencias medibles.
    """
    if x <= 0: return 0
    resultado = 0
    # El ciclo interno multiplica el esfuerzo de la CPU para diagnóstico experimental
    for _ in range(100):
        resultado += (math.sqrt(x) * math.tan(x) / math.log1p(x)) + math.exp(math.sin(x))
    return resultado / 5


# 2. IMPLEMENTACIÓN DE VARIANTES DE PROCESAMIENTO
def procesar_secuencial(lista_datos):
    """Versión que procesa elemento a elemento."""
    total = 0.0
    for x in lista_datos:
        total += carga_pesada(x)
    return total

def procesar_por_bloques(lista_datos, tam_bloque=50_000):
    """Versión que procesa la lista en bloques para observar independencia."""
    total_final = 0.0
    for i in range(0, len(lista_datos), tam_bloque):
        bloque = lista_datos[i : i + tam_bloque]
        subtotal = sum(carga_pesada(x) for x in bloque)
        total_final += subtotal
    return total_final


# 3. EJECUCIÓN Y MEDICIÓN (DATOS DEL LABORATORIO)
# Carga de los 400,000 datos generados previamente
try:
    df = pd.read_csv('datos_ejercicio2.csv')
    datos = df['Numeros'].tolist()
except FileNotFoundError:
    print("Error: No se encontró el archivo 'datos_ejercicio2.csv'.")
    datos = list(range(1, 400_001)) # Fallback si no existe el archivo

# Medición Variante Secuencial
inicio_sec = time.perf_counter()
res_sec = procesar_secuencial(datos)
tiempo_sec = time.perf_counter() - inicio_sec

# Medición Variante por Bloques
inicio_bloq = time.perf_counter()
res_bloq = procesar_por_bloques(datos, tam_bloque=50_000)
tiempo_bloq = time.perf_counter() - inicio_bloq


# 4. PRESENTACIÓN DE RESULTADOS CON SYMPY
# Configuración de la fórmula matemática elegante
x_sym = sp.symbols('x')
f_x = (sp.sqrt(x_sym) * sp.tan(x_sym) / sp.ln(x_sym + 1)) + sp.exp(sp.sin(x_sym))

print("Función de transformación costosa utilizada:")
display(Math(f"f(x) = {sp.latex(f_x)}"))

print("-" * 75)
print(f"{'Variante':<20} | {'Resultado Total':<25} | {'Tiempo (s)':<12}")
print("-" * 75)
print(f"{'Secuencial':<20} | {res_sec:25.4f} | {tiempo_sec:12.6f}")
print(f"{'Por Bloques':<20} | {res_bloq:25.4f} | {tiempo_bloq:12.6f}")
print("-" * 75)

Función de transformación costosa utilizada:


<IPython.core.display.Math object>

---------------------------------------------------------------------------
Variante             | Resultado Total           | Tiempo (s)  
---------------------------------------------------------------------------
Secuencial           |           -348375536.3519 |    14.142199
Por Bloques          |           -348375536.3519 |    14.135963
---------------------------------------------------------------------------


La variante por bloques demuestra que el problema posee unidades de trabajo independientes. Dado que el tiempo de ejecución secuencial es significativo (~14.1 segundos) y la carga de trabajo es intensiva en CPU debido a la función $f(x)$, este problema es un candidato ideal para la paralelización por datos. La implementación por bloques ya establece la infraestructura lógica necesaria para distribuir la carga en múltiples núcleos en una etapa posterior.

La igualdad en los resultados totales confirma que la segmentación por bloques mantiene la precisión del cálculo original.

Imaginemos un escenario de una plataforma de streaming (como Netflix o Spotify) donde debemos procesar datos de consumo de 8 regiones diferentes. Cada "lote" de este ejercicio representa una región (Chile, Argentina, Perú, etc.).
* Enfoque Secuencial (Estado Actual): Es equivalente a tener un solo analista para todo el continente. El analista procesa Chile, luego Argentina, y así sucesivamente. El tiempo total es la suma de los tiempos individuales.
* Enfoque Paralelo (Oportunidad): Equivale a contratar 8 analistas (uno por cada núcleo del procesador). Como los datos de cada región son independientes, todos pueden trabajar simultáneamente, reduciendo el tiempo total al tiempo que tarda la región más pesada.

## EJERCICIO 3: Caso aplicado a ciencia de datos y decisión diagnóstica.

In [ ]:
# 1. Simulación de 8 Regiones con datos aleatorios (Variabilidad)
random.seed(42)
regiones = ["Norte", "Sur", "Centro", "Este", "Oeste", "Costa", "Cordillera", "Insular"]
lotes_regionales = []

print("Generando datos regionales (8 millones de registros aleatorios)...")
for _ in range(8):
    # Generamos datos aleatorios para que las medias sean distintas
    lote = [random.uniform(0, 1000) for _ in range(1_000_000)]
    lotes_regionales.append(lote)

# 2. Función de "Transformación de Alta Intensidad"
def procesar_datos_regionales(lote):
    """
    Aumentamos la carga de procesamiento para que el tiempo sea medible.
    Simula una limpieza profunda de datos.
    """
    # Procesamos 500,000 elementos con operaciones trigonométricas pesadas
    resultado_limpio = []
    for x in lote[:500_000]:
        # Esta operación obliga a la CPU a trabajar mucho más por cada dato
        val = (math.sqrt(x) * math.tan(x) / math.log1p(x)) + math.exp(math.sin(x))
        resultado_limpio.append(val)

    return {
        "media_transformada": statistics.fmean(resultado_limpio),
        "max": max(resultado_limpio),
        "min": min(resultado_limpio)
    }

# 3. Ejecución del Diagnóstico
print("\nIniciando procesamiento secuencial por regiones...")
inicio = time.perf_counter()
reportes = []

for nombre, datos in zip(regiones, lotes_regionales):
    print(f"-> Procesando región: {nombre}...")
    resumen = procesar_datos_regionales(datos)
    reportes.append((nombre, resumen))

fin = time.perf_counter()
tiempo_total = fin - inicio

# 4. Tabla de Resultados Final
print("\n" + "="*85)
print(f"{'REGIÓN':<15} | {'MEDIA TRANSF.':<18} | {'MIN':<12} | {'MAX':<12}")
print("-" * 85)
for nombre, r in reportes:
    print(f"{nombre:<15} | {r['media_transformada']:<18.4f} | {r['min']:<12.2f} | {r['max']:<12.2f}")

print("-" * 85)
print(f"TIEMPO TOTAL SECUENCIAL: {tiempo_total:.4f} segundos")
print("="*85)

Generando datos regionales (8 millones de registros aleatorios)...

Iniciando procesamiento secuencial por regiones...
-> Procesando región: Norte...
-> Procesando región: Sur...
-> Procesando región: Centro...
-> Procesando región: Este...
-> Procesando región: Oeste...
-> Procesando región: Costa...
-> Procesando región: Cordillera...
-> Procesando región: Insular...

REGIÓN          | MEDIA TRANSF.      | MIN          | MAX         
-------------------------------------------------------------------------------------
Norte           | 1.7887             | -1343922.60  | 2057054.75  
Sur             | -57.9764           | -28820275.52 | 354721.94   
Centro          | 1.7501             | -286152.67   | 473420.34   
Este            | 1.6945             | -276353.20   | 165929.71   
Oeste           | -5.3679            | -3000321.83  | 352453.66   
Costa           | 1.7924             | -277167.51   | 238247.40   
Cordillera      | -4.8881            | -1669174.80  | 1335071.22  
Insul

- ANÁLISIS DE LOS RESULTADOS

Se observa que las medias (como **1.7887** vs **-64.1713**) y los rangos (MIN/MAX) son distintos para cada región. Esto confirma que se estan procesando **unidades de trabajo independientes** con datos heterogéneos, simulando un caso real de ciencia de datos.

Los **1.52 segundos** es un tiempo frontera. Para un solo proceso no es mucho, pero en un flujo de trabajo donde este cálculo se repite miles de veces o con volúmenes 100 veces mayores, se convierte en un cuello de botella crítico.

- DECISIÓN DIAGNÓSTICA Y JUSTIFICACIÓN:

**Decisión:** **Computación Paralela en un mismo equipo**.

**Justificación Técnica:**

 El problema se divide en 8 lotes que no comparten memoria ni estados entre sí. Esto lo hace ideal para el paralelismo de datos.

 Con un tiempo de 1.52s para solo 8 regiones, el beneficio de usar múltiples núcleos es claro. Si tu CPU tiene 8 hilos, podrías reducir este tiempo a cerca de **0.2 segundos**.

 Los 8 millones de registros caben en la RAM local. Por lo tanto, la **Computación Distribuida** se descarta, ya que el tiempo que tardarías en enviar los datos por la red a otras computadoras sería mayor al tiempo de procesamiento local (latencia de red).

- REFLEXIÓN CONECTADA CON CIENCIA DE DATOS:

En la formación de ciencia de datos, este ejercicio demuestra que no basta con que un programa "funcione". Identificar que cada región es un lote independiente permite diseñar arquitecturas escalables. Por ejemplo, en una **validación cruzada** o en el **preprocesamiento masivo de logs**, aplicar paralelismo en el mismo equipo es la forma más eficiente de optimizar recursos sin aumentar la complejidad de la infraestructura.

## REFLEXIÓN FINAL

Para que el paralelismo sea viable, el problema debe presentar independencia de datos o de tareas. Como observamos en el Ejercicio 2 y 3, es fundamental que las unidades de trabajo (bloques o regiones) no dependan de los resultados de las otras para ejecutarse simultáneamente. Además, la carga computacional debe ser lo suficientemente alta para que la ganancia en tiempo compense el costo de gestionar múltiples hilos.

Una demora no justifica por sí sola la paralelización. Si el retraso se debe a una tarea estrictamente secuencial (como la suma acumulada del Ejercicio 1), añadir más núcleos no reducirá el tiempo debido a las dependencias de datos. Además, si el volumen de datos es pequeño, el costo de coordinar el paralelismo (overhead) podría terminar haciendo el programa más lento que la versión original.

Basado en lo aprendido, aportaría valor en tareas de alta intensidad repetitiva. Ejemplos claros son la validación cruzada (entrenar modelos en diferentes particiones de forma simultánea), el preprocesamiento masivo de imágenes o logs regionales, y simulaciones de Monte Carlo, donde cada iteración es independiente de la anterior.

En este trabajo se aprendió que la medición es la única forma de establecer una línea base objetiva. Sin datos (como los 14.1s del Ejercicio 2), no podríamos saber si una mejora es real o si estamos introduciendo complejidad innecesaria. La medición permite identificar si el cuello de botella es la CPU, la memoria o la red, guiando la decisión hacia la estrategia correcta (secuencial, paralela o distribuida).